# Notebook 6 - Phi-2 Colab Runtime Notes

This notebook keeps the project Phi-2-only. It is useful when local hardware is too slow for activation extraction and a CUDA Colab runtime is available.


## Runtime Setup

Select a GPU runtime before running this notebook. The code below installs the package in editable mode if the notebook is running from a cloned repository.


In [ ]:
# In Colab, uncomment these lines after cloning the repository.
# %pip install -r requirements.txt
# %pip install -e .


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_DIR = PROJECT_ROOT / "results"
ACTIVATION_CACHE_DIR = PROJECT_ROOT / "data" / "activations"
RESULTS_DIR.mkdir(exist_ok=True)
ACTIVATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Minimal Phi-2 Run

This small run checks that datasets, activation extraction, probe training, and evaluation work end to end.


In [ ]:
import matplotlib.pyplot as plt

from lie_detector_llm.datasets import build_dataset_collection
from lie_detector_llm.experiment import DEFAULT_MODEL, run_full_transfer_matrix
from lie_detector_llm.plotting import plot_transfer_heatmap

DATASET_NAMES = ["facts", "dbpedia_14", "amazon_polarity", "repeng_truthful"]
MAX_GROUPS = 30
LAYER_INDEX = 18
ACTIVATION_BATCH_SIZE = 2
MAX_LENGTH = 512
LOAD_IN_4BIT = False

collection = build_dataset_collection(
    dataset_names=DATASET_NAMES,
    max_groups=MAX_GROUPS,
    seed=0,
)
display(collection.summary())

matrix = run_full_transfer_matrix(
    collection=collection,
    model_name=DEFAULT_MODEL,
    probe_method="dim",
    layer_index=LAYER_INDEX,
    activation_batch_size=ACTIVATION_BATCH_SIZE,
    max_length=MAX_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    show_progress=True,
    activation_cache_dir=ACTIVATION_CACHE_DIR,
)

fig, ax = plot_transfer_heatmap(
    matrix.results,
    title=f"Phi-2 transfer matrix, DIM probe, layer {LAYER_INDEX}",
)
plt.show()


## Discussion

Use this notebook only to run the same Phi-2 experiment on a faster GPU. Do not add other base models here if the report objective is the strict Phi-2 reproduction.
